# 03 — Transaction Inactivity Prediction Model

## FinPulse AI — Temporal Binary Classification

This notebook trains and evaluates machine learning models to predict future customer transactional inactivity.

The objective is to estimate whether an eligible customer will perform no transactions during the 90 days following a quarterly cutoff date.

The binary target is defined as:

- `0`: the customer performs at least one transaction during the following 90 days;
- `1`: the customer performs no transactions during the following 90 days.

Each prediction uses only transactional information available during the 180 days preceding the cutoff date.

The temporal dataset was produced by the `02_churn_temporal_dataset.ipynb` notebook and includes chronological train, validation, test and embargo partitions.

To prevent temporal leakage:

- models are trained exclusively on the training partition;
- model selection is performed using the validation partition;
- embargo periods are excluded from model training and evaluation;
- the test partition remains untouched until the final model is selected;
- customer identifiers and cutoff dates are not used as predictive features.

The modeling workflow includes:

1. dataset and metadata validation;
2. baseline model construction;
3. feature redundancy analysis;
4. model training and comparison;
5. threshold evaluation;
6. experiment tracking with MLflow;
7. final evaluation on the temporal test partition;
8. registration of the selected model.

## 1. Dataset Loading and Validation

The temporal dataset and its metadata are loaded from the processed data layer.

This section validates the dataset schema, feature definitions, target integrity and chronological partitions before any model is trained.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_PATH = Path(
    "/home/jovyan/data/processed"
)

DATASET_FILE = (
    PROCESSED_PATH
    / "customer_transaction_inactivity_temporal_dataset.parquet"
)

METADATA_FILE = (
    PROCESSED_PATH
    / "customer_transaction_inactivity_metadata.json"
)

assert DATASET_FILE.exists(), (
    f"Dataset não encontrado: {DATASET_FILE}"
)

assert METADATA_FILE.exists(), (
    f"Metadados não encontrados: {METADATA_FILE}"
)

temporal_dataset = pd.read_parquet(
    DATASET_FILE
)

with open(
    METADATA_FILE,
    "r",
    encoding="utf-8"
) as metadata_file:
    dataset_metadata = json.load(metadata_file)

TARGET = dataset_metadata["target"]
FEATURE_COLUMNS = dataset_metadata["features"]

IDENTIFIER_COLUMNS = [
    "customer_id",
    "cutoff_date"
]

temporal_dataset["cutoff_date"] = pd.to_datetime(
    temporal_dataset["cutoff_date"]
)

print("Dataset carregado!")
print("Shape:", temporal_dataset.shape)
print("Target:", TARGET)
print("Features:", len(FEATURE_COLUMNS))
print("Cutoffs:", temporal_dataset["cutoff_date"].nunique())

Dataset carregado!
Shape: (658587, 22)
Target: transaction_inactivity_90d_label
Features: 18
Cutoffs: 22


## 2. Temporal Modeling Partitions

The training and validation partitions are selected according to the temporal split created in the previous notebook.

The embargo observations are excluded, and the test partition is reserved exclusively for the final evaluation of the selected model.

In [2]:
required_columns = (
    IDENTIFIER_COLUMNS
    + ["dataset_split"]
    + FEATURE_COLUMNS
    + [TARGET]
)

missing_columns = [
    column
    for column in required_columns
    if column not in temporal_dataset.columns
]

assert missing_columns == [], (
    f"Colunas ausentes: {missing_columns}"
)

assert temporal_dataset.duplicated(
    subset=IDENTIFIER_COLUMNS
).sum() == 0

assert temporal_dataset[
    FEATURE_COLUMNS
].isna().sum().sum() == 0

assert np.isinf(
    temporal_dataset[FEATURE_COLUMNS]
).sum().sum() == 0

assert set(
    temporal_dataset[TARGET].unique()
) == {0, 1}

assert set(
    temporal_dataset["dataset_split"].unique()
) == {
    "train",
    "validation",
    "test",
    "embargo"
}

print("Input dataset validation passed!")

Input dataset validation passed!


In [3]:
train_data = (
    temporal_dataset[
        temporal_dataset["dataset_split"] == "train"
    ]
    .copy()
)

validation_data = (
    temporal_dataset[
        temporal_dataset["dataset_split"] == "validation"
    ]
    .copy()
)

X_train = train_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET].astype("int8").copy()

X_validation = (
    validation_data[FEATURE_COLUMNS]
    .copy()
)

y_validation = (
    validation_data[TARGET]
    .astype("int8")
    .copy()
)

print("Treino:", X_train.shape)
print("Validação:", X_validation.shape)

print(
    "Taxa de inatividade no treino:",
    f"{y_train.mean():.2%}"
)

print(
    "Taxa de inatividade na validação:",
    f"{y_validation.mean():.2%}"
)

assert train_data["cutoff_date"].max() < (
    validation_data["cutoff_date"].min()
)

print("Modeling split validation passed!")

Treino: (448919, 18)
Validação: (89934, 18)
Taxa de inatividade no treino: 41.39%
Taxa de inatividade na validação: 41.43%
Modeling split validation passed!


## 3. Baseline Model

Before training predictive models, a naive baseline is established using `DummyClassifier`.

The baseline does not learn transactional patterns. It predicts according to the class distribution observed in the training data.

Every trained model must outperform this baseline, particularly in:

- average precision;
- ROC AUC;
- recall for inactive customers;
- F1-score;
- balanced accuracy.

All baseline metrics are calculated exclusively on the temporal validation partition.

In [4]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

def evaluate_binary_classifier(
    y_true,
    probabilities,
    threshold=0.50
):
    predictions = (
        probabilities >= threshold
    ).astype("int8")

    metrics = {
        "threshold": float(threshold),
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
        "average_precision": average_precision_score(
            y_true,
            probabilities
        ),
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_true,
            predictions,
            zero_division=0
        )
    }

    confusion = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    )

    confusion_frame = pd.DataFrame(
        confusion,
        index=[
            "actual_active",
            "actual_inactive"
        ],
        columns=[
            "predicted_active",
            "predicted_inactive"
        ]
    )

    return metrics, confusion_frame

In [5]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42
)

dummy_model.fit(
    X_train,
    y_train
)

dummy_validation_probabilities = (
    dummy_model.predict_proba(
        X_validation
    )[:, 1]
)

dummy_metrics, dummy_confusion = (
    evaluate_binary_classifier(
        y_validation,
        dummy_validation_probabilities,
        threshold=0.50
    )
)

dummy_metrics_frame = pd.DataFrame(
    [dummy_metrics],
    index=["dummy_classifier"]
).round(4)

display(dummy_metrics_frame)
display(dummy_confusion)

,threshold,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score
dummy_classifier,0.5,0.5,0.4143,0.5857,0.5,0.0,0.0,0.0


,predicted_active,predicted_inactive
actual_active,52675,0
actual_inactive,37259,0


## 4. Logistic Regression Baseline

Logistic regression is used as the first trainable baseline.

Unlike the dummy model, it learns relationships between historical transactional features and future inactivity.

A preprocessing pipeline standardizes the numerical features using statistics calculated exclusively from the training partition. The validation partition is transformed using the same fitted scaler, preventing information leakage.

The initial classification threshold is fixed at `0.50`. Threshold optimization will be performed only after the models are compared.

In [6]:
import time

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

logistic_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

training_start = time.perf_counter()

logistic_model.fit(
    X_train,
    y_train
)

logistic_training_seconds = (
    time.perf_counter() - training_start
)

logistic_validation_probabilities = (
    logistic_model.predict_proba(
        X_validation
    )[:, 1]
)

logistic_metrics, logistic_confusion = (
    evaluate_binary_classifier(
        y_validation,
        logistic_validation_probabilities,
        threshold=0.50
    )
)

logistic_metrics[
    "training_seconds"
] = logistic_training_seconds

print(
    "Tempo de treinamento:",
    f"{logistic_training_seconds:.2f} segundos"
)

display(
    pd.DataFrame(
        [logistic_metrics],
        index=["logistic_regression"]
    ).round(4)
)

display(logistic_confusion)

Tempo de treinamento: 3.43 segundos


,threshold,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score,training_seconds
logistic_regression,0.5,0.5974,0.4834,0.5865,0.52,0.5035,0.1321,0.2093,3.4328


,predicted_active,predicted_inactive
actual_active,47822,4853
actual_inactive,32337,4922


In [7]:
model_comparison = pd.DataFrame(
    [
        {
            "model": "dummy_classifier",
            **dummy_metrics
        },
        {
            "model": "logistic_regression",
            **logistic_metrics
        }
    ]
)

comparison_columns = [
    "model",
    "roc_auc",
    "average_precision",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1_score",
    "threshold"
]

model_comparison = (
    model_comparison[comparison_columns]
    .sort_values(
        "average_precision",
        ascending=False
    )
    .reset_index(drop=True)
)

display(model_comparison.round(4))

,model,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score,threshold
0,logistic_regression,0.5974,0.4834,0.5865,0.52,0.5035,0.1321,0.2093,0.5
1,dummy_classifier,0.5000,0.4143,0.5857,0.50,0.0000,0.0000,0.0000,0.5


## 5. Feature Redundancy Analysis

Highly correlated features may represent nearly identical transactional information.

This section identifies strong pairwise relationships using only the training partition. The target and validation data are not used in this analysis.

Correlated features are not removed automatically. Each pair will be evaluated according to its meaning, stability and usefulness for subsequent models.

In [8]:
train_feature_correlations = (
    X_train
    .corr(method="spearman")
    .abs()
)

upper_triangle = (
    train_feature_correlations
    .where(
        np.triu(
            np.ones(
                train_feature_correlations.shape
            ),
            k=1
        ).astype(bool)
    )
)

high_correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "absolute_spearman_correlation"
]

high_correlation_pairs = (
    high_correlation_pairs[
        high_correlation_pairs[
            "absolute_spearman_correlation"
        ] >= 0.90
    ]
    .sort_values(
        "absolute_spearman_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Pares com correlação maior ou igual a 0.90:",
    len(high_correlation_pairs)
)

display(high_correlation_pairs)

Pares com correlação maior ou igual a 0.90: 10


,feature_1,feature_2,absolute_spearman_correlation
0,transactions_last_180d,merchant_diversity_last_180d,0.999905
1,transaction_change_30d,transaction_change_rate_30d,0.997664
2,transactions_last_180d,active_days_last_180d,0.997556
3,active_days_last_180d,merchant_diversity_last_180d,0.997460
4,transactions_last_30d,amount_last_30d,0.979470
5,transactions_previous_30d,amount_previous_30d,0.979114
6,transactions_last_30d,recent_transaction_share,0.970410
7,amount_last_30d,recent_transaction_share,0.955175
8,transaction_change_rate_30d,amount_change_30d,0.924242
9,transaction_change_30d,amount_change_30d,0.922931


### 5.1 Reduced Feature Candidate

The redundancy analysis indicates that several engineered features describe nearly identical historical behavior.

A reduced feature candidate is created by removing selected derived or highly redundant variables.

Transaction amount features are retained even when correlated with transaction counts because they represent a different business concept and may become more informative when the platform receives more realistic data.

The original feature set remains unchanged. Both versions will be evaluated on the same temporal validation partition.

In [9]:
REDUNDANT_FEATURES_TO_DROP = [
    "merchant_diversity_last_180d",
    "active_days_last_180d",
    "transaction_change_rate_30d",
    "amount_change_30d",
    "recent_transaction_share"
]

REDUCED_FEATURE_COLUMNS = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in REDUNDANT_FEATURES_TO_DROP
]

print(
    "Features originais:",
    len(FEATURE_COLUMNS)
)

print(
    "Features reduzidas:",
    len(REDUCED_FEATURE_COLUMNS)
)

print("\nFeatures removidas:")

for feature in REDUNDANT_FEATURES_TO_DROP:
    print("-", feature)

Features originais: 18
Features reduzidas: 13

Features removidas:
- merchant_diversity_last_180d
- active_days_last_180d
- transaction_change_rate_30d
- amount_change_30d
- recent_transaction_share


In [10]:
X_train_reduced = (
    train_data[REDUCED_FEATURE_COLUMNS]
    .copy()
)

X_validation_reduced = (
    validation_data[REDUCED_FEATURE_COLUMNS]
    .copy()
)

logistic_reduced_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

training_start = time.perf_counter()

logistic_reduced_model.fit(
    X_train_reduced,
    y_train
)

logistic_reduced_training_seconds = (
    time.perf_counter() - training_start
)

logistic_reduced_probabilities = (
    logistic_reduced_model.predict_proba(
        X_validation_reduced
    )[:, 1]
)

(
    logistic_reduced_metrics,
    logistic_reduced_confusion
) = evaluate_binary_classifier(
    y_validation,
    logistic_reduced_probabilities,
    threshold=0.50
)

logistic_reduced_metrics[
    "training_seconds"
] = logistic_reduced_training_seconds

display(
    pd.DataFrame(
        [logistic_reduced_metrics],
        index=["logistic_reduced"]
    ).round(4)
)

display(logistic_reduced_confusion)

,threshold,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score,training_seconds
logistic_reduced,0.5,0.5973,0.4833,0.5866,0.5194,0.5041,0.1274,0.2034,1.2701


,predicted_active,predicted_inactive
actual_active,48006,4669
actual_inactive,32513,4746


In [11]:
logistic_feature_comparison = pd.DataFrame([
    {
        "model": "logistic_full",
        "features": len(FEATURE_COLUMNS),
        **logistic_metrics
    },
    {
        "model": "logistic_reduced",
        "features": len(REDUCED_FEATURE_COLUMNS),
        **logistic_reduced_metrics
    }
])

display(
    logistic_feature_comparison[
        [
            "model",
            "features",
            "roc_auc",
            "average_precision",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1_score",
            "training_seconds"
        ]
    ]
    .sort_values(
        "average_precision",
        ascending=False
    )
    .round(4)
)

,model,features,roc_auc,average_precision,balanced_accuracy,precision,recall,f1_score,training_seconds
0,logistic_full,18,0.5974,0.4834,0.5200,0.5035,0.1321,0.2093,3.4328
1,logistic_reduced,13,0.5973,0.4833,0.5194,0.5041,0.1274,0.2034,1.2701


## 6. Nonlinear Tree-Based Baseline

A histogram-based gradient boosting classifier is trained as a nonlinear CPU baseline.

Unlike logistic regression, this model can learn nonlinear relationships and interactions between historical transactional features.

The reduced feature set is used to limit redundancy. Feature scaling is not required for this tree-based model.

Early stopping is disabled because model selection is performed using the predefined temporal validation partition.

In [19]:
from sklearn.ensemble import (
    HistGradientBoostingClassifier
)

hist_gradient_model = (
    HistGradientBoostingClassifier(
        learning_rate=0.10,
        max_iter=200,
        max_leaf_nodes=31,
        min_samples_leaf=50,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42
    )
)

training_start = time.perf_counter()

hist_gradient_model.fit(
    X_train_reduced,
    y_train
)

hist_gradient_training_seconds = (
    time.perf_counter() - training_start
)

print(
    "Tempo de treinamento:",
    f"{hist_gradient_training_seconds:.2f} segundos"
)

Tempo de treinamento: 3.48 segundos


In [20]:
 hist_gradient_probabilities = (
    hist_gradient_model.predict_proba(
        X_validation_reduced
    )[:, 1]
)

(
    hist_gradient_metrics,
    hist_gradient_confusion
) = evaluate_binary_classifier(
    y_validation,
    hist_gradient_probabilities,
    threshold=0.50
)

hist_gradient_metrics[
    "training_seconds"
] = hist_gradient_training_seconds

display(
    pd.DataFrame(
        [hist_gradient_metrics],
        index=["hist_gradient_boosting"]
    ).round(4)
)

display(hist_gradient_confusion)

,threshold,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score,training_seconds
hist_gradient_boosting,0.5,0.597,0.4816,0.5859,0.5318,0.5006,0.2163,0.3021,3.4845


,predicted_active,predicted_inactive
actual_active,44636,8039
actual_inactive,29200,8059


In [12]:
!nvidia-smi

Wed Jul 15 21:16:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.45.04              Driver Version: 595.76         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060        On  |   00000000:10:00.0  On |                  N/A |
|  0%   33C    P5             14W /  155W |    2543MiB /   8151MiB |     10%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import sys

print("Python do kernel:", sys.executable)

%pip show xgboost

Python do kernel: /opt/conda/bin/python
Name: xgboost
Version: 3.2.0
Summary: XGBoost Python Package
Home-page: 
Author: 
Author-email: Hyunsu Cho <chohyu01@cs.washington.edu>, Jiaming Yuan <jm.yuan@outlook.com>
License: Apache-2.0
Location: /opt/conda/lib/python3.11/site-packages
Requires: numpy, nvidia-nccl-cu12, scipy
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [14]:
import sys

!{sys.executable} -m pip install --no-cache-dir xgboost==3.2.0

In [15]:
from xgboost import XGBClassifier

xgboost_gpu_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",

    n_estimators=1500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,

    subsample=0.80,
    colsample_bytree=0.80,

    reg_alpha=0.10,
    reg_lambda=1.00,

    tree_method="hist",
    device="cuda",

    early_stopping_rounds=75,
    random_state=42
)

training_start = time.perf_counter()

xgboost_gpu_model.fit(
    X_train_reduced,
    y_train,
    eval_set=[
        (
            X_validation_reduced,
            y_validation
        )
    ],
    verbose=50
)

xgboost_training_seconds = (
    time.perf_counter() - training_start
)

print(
    "Tempo de treinamento:",
    f"{xgboost_training_seconds:.2f} segundos"
)

print(
    "Melhor iteração:",
    xgboost_gpu_model.best_iteration
)

print(
    "Melhor score:",
    xgboost_gpu_model.best_score
)

[0]	validation_0-aucpr:0.46478
[50]	validation_0-aucpr:0.48299
[86]	validation_0-aucpr:0.48259
Tempo de treinamento: 1.49 segundos
Melhor iteração: 11
Melhor score: 0.4838529947536464


In [16]:
xgboost_validation_probabilities = (
    xgboost_gpu_model.predict_proba(
        X_validation_reduced
    )[:, 1]
)

(
    xgboost_metrics,
    xgboost_confusion
) = evaluate_binary_classifier(
    y_validation,
    xgboost_validation_probabilities,
    threshold=0.50
)

xgboost_metrics[
    "training_seconds"
] = xgboost_training_seconds

display(
    pd.DataFrame(
        [xgboost_metrics],
        index=["xgboost_gpu"]
    ).round(4)
)

display(xgboost_confusion)

/opt/conda/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [21:17:27] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,threshold,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score,training_seconds
xgboost_gpu,0.5,0.598,0.4838,0.5857,0.5,0.0,0.0,0.0,1.4931


,predicted_active,predicted_inactive
actual_active,52675,0
actual_inactive,37259,0


In [21]:
gpu_model_comparison = pd.DataFrame([
    {
        "model": "dummy_classifier",
        "features": len(FEATURE_COLUMNS),
        **dummy_metrics
    },
    {
        "model": "logistic_reduced",
        "features": len(REDUCED_FEATURE_COLUMNS),
        **logistic_reduced_metrics
    },
    {
        "model": "hist_gradient_boosting",
        "features": len(REDUCED_FEATURE_COLUMNS),
        **hist_gradient_metrics
    },
    {
        "model": "xgboost_gpu",
        "features": len(REDUCED_FEATURE_COLUMNS),
        **xgboost_metrics
    }
])

display(
    gpu_model_comparison[
        [
            "model",
            "features",
            "roc_auc",
            "average_precision",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1_score",
            "training_seconds"
        ]
    ]
    .sort_values(
        "average_precision",
        ascending=False
    )
    .reset_index(drop=True)
    .round(4)
)

,model,features,roc_auc,average_precision,balanced_accuracy,precision,recall,f1_score,training_seconds
0,xgboost_gpu,13,0.5980,0.4838,0.5000,0.0000,0.0000,0.0000,1.4931
1,logistic_reduced,13,0.5973,0.4833,0.5194,0.5041,0.1274,0.2034,1.2701
2,hist_gradient_boosting,13,0.5970,0.4816,0.5318,0.5006,0.2163,0.3021,3.4845
3,dummy_classifier,18,0.5000,0.4143,0.5000,0.0000,0.0000,0.0000,NaN


## 8. GPU-Accelerated CatBoost

CatBoost provides a second GPU-accelerated gradient boosting implementation.

It is evaluated to determine whether its ordered boosting strategy captures temporal transaction patterns that were not captured by XGBoost.

The temporal validation partition is used for early stopping, while the test partition remains untouched.

In [22]:
import sys

!{sys.executable} -m pip install --no-cache-dir catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 MB 11.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 11.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.0/461.0 kB 14.4 MB/s eta 0:00:00


In [23]:
from catboost import CatBoostClassifier

catboost_gpu_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=7,

    loss_function="Logloss",
    eval_metric="PRAUC:type=Classic",

    l2_leaf_reg=5.0,
    random_strength=1.0,
    border_count=128,

    task_type="GPU",
    devices="0",

    random_seed=42,
    allow_writing_files=False,
    verbose=100
)

training_start = time.perf_counter()

catboost_gpu_model.fit(
    X_train_reduced,
    y_train,
    eval_set=(
        X_validation_reduced,
        y_validation
    ),
    early_stopping_rounds=75,
    use_best_model=True
)

catboost_training_seconds = (
    time.perf_counter() - training_start
)

print(
    "Tempo de treinamento:",
    f"{catboost_training_seconds:.2f} segundos"
)

print(
    "Melhor iteração:",
    catboost_gpu_model.get_best_iteration()
)

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.5908695	test: 0.5908274	best: 0.5908274 (0)	total: 14.8s	remaining: 6h 10m 46s
bestTest = 0.5908274277
bestIteration = 0
Shrink model to first 1 iterations.
Tempo de treinamento: 27.54 segundos
Melhor iteração: 0


In [24]:
catboost_validation_probabilities = (
    catboost_gpu_model.predict_proba(
        X_validation_reduced
    )[:, 1]
)

(
    catboost_metrics,
    catboost_confusion
) = evaluate_binary_classifier(
    y_validation,
    catboost_validation_probabilities,
    threshold=0.50
)

catboost_metrics[
    "training_seconds"
] = catboost_training_seconds

display(
    pd.DataFrame(
        [catboost_metrics],
        index=["catboost_gpu"]
    ).round(4)
)

display(catboost_confusion)

,threshold,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1_score,training_seconds
catboost_gpu,0.5,0.5971,0.4762,0.5863,0.5643,0.5008,0.4359,0.4661,27.5353


,predicted_active,predicted_inactive
actual_active,36487,16188
actual_inactive,21018,16241
